[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/06_Floating_Point.ipynb)

# Floating Point and Numerical Error

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 5

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

## Learning goals

By the end of this notebook you should be able to:

- explain how integers and real numbers are stored in binary, and why `0.1 + 0.2 != 0.3`;
- describe a 64-bit float as sign, exponent, and mantissa, and state its precision (machine epsilon) and range;
- recognize overflow, underflow, `inf`, and `nan`, and test floats for equality safely with `np.isclose`;
- compute absolute and relative errors and read them on a log-log plot;
- spot **catastrophic cancellation** and rewrite formulas to avoid it (`1 - cos x`, $\gamma - 1$, the quadratic formula, `expm1`, `log1p`);
- distinguish **round-off error** from **truncation error**.

Physics along the way: the kinetic energy of a slow electron, Boltzmann factors, and Taylor series. This notebook is the foundation for everything in Unit 2: every numerical method you meet (derivatives, integrals, ODEs) has both kinds of error, and next notebook you will see them fight each other.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants as const

## 1. A surprise

Try this:

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

This is not a bug in Python. Every language that uses the computer's standard floating-point numbers (C, Fortran, Java, JavaScript, Excel...) gives the same answer. To understand why, we need to look at how numbers are stored.

## 2. Numbers in binary

### 2.1 Integers

A computer stores everything as **bits**, 0s and 1s. An integer is written in base 2, the way you write numbers in base 10 but with powers of 2:

$$13 = 1\cdot 2^3 + 1\cdot 2^2 + 0\cdot 2^1 + 1\cdot 2^0 = 1101_2.$$

Python's `bin()` shows the binary form (with a `0b` prefix), and `int(string, 2)` converts back.

In [ ]:
for n in [1, 2, 5, 13, 255, 1024]:
    print(f"{n:5d} = {bin(n)}")
print(int("1101", 2))

Integers are stored **exactly**. Python's own `int` can even grow as large as memory allows:

In [ ]:
print(2**200)

NumPy integers, however, have a fixed size, usually 64 bits (`int64`), so the largest is $2^{63}-1 \approx 9.2\times10^{18}$. Going past it **wraps around** to negative numbers, silently in array arithmetic. This rarely matters in physics, but it is a nasty surprise when it does (e.g. counting events, or computing factorials with integer arrays).

In [ ]:
big = np.array([2**62], dtype=np.int64)
print(np.iinfo(np.int64).max)
print(big * 2)     # wraps around: no error, wrong answer!

### 2.2 Fractions in binary

Fractions use negative powers of 2: $0.5 = 2^{-1} = 0.1_2$, $0.75 = 2^{-1} + 2^{-2} = 0.11_2$. These are exact. But some fractions never terminate. You know $1/3 = 0.3333\ldots$ in base 10. In base 2, $1/10$ is a repeating pattern:

$$0.1 = 0.0001100110011001100\ldots_2$$

The computer has only a fixed number of bits, so it cuts this off and stores the **nearest representable number**. Printing more digits than Python normally shows reveals it:

In [ ]:
print(f"{0.1:.25f}")
print(f"{0.2:.25f}")
print(f"{0.3:.25f}")
print(f"{0.1 + 0.2:.25f}")
print(f"{0.5:.25f}")      # exactly representable

`0.1` and `0.2` are both stored slightly too large, and their sum lands on a different float than the one nearest to `0.3`. The difference is tiny, about $5\times10^{-17}$, but `==` demands *exact* equality.

**Lesson:** almost every real number you type or compute is stored with a small error. Usually that error is harmless, but there are specific situations, which this notebook is about, where it grows until it ruins the answer.

## 3. How a float is stored

A Python `float` (NumPy `float64`, "double precision") is stored in 64 bits, using the binary version of scientific notation. Just as $6.022\times10^{23}$ has a sign, a mantissa (6.022), and an exponent (23), a float is

$$x = (-1)^{s} \times 1.f_1 f_2 \ldots f_{52} {}_{(2)} \times 2^{\,e - 1023}$$

with

- **1 sign bit** $s$ (0 for positive, 1 for negative);
- **11 exponent bits** $e$, giving powers of 2 from about $2^{-1022}$ to $2^{1023}$ (the "$-1023$" offset lets the exponent be negative);
- **52 mantissa (fraction) bits** $f_i$. The leading "1." is not stored, since it is always there.

The mantissa sets the **precision** (how many significant digits); the exponent sets the **range** (how large or small). You do not need to memorize the layout, but it explains everything that follows. The function below shows the three parts of any float. (It uses a trick, `.view`, to reinterpret the 64 bits as an integer; you do not need to understand that line.)

In [ ]:
def float_bits(x):
    """Return the sign, exponent, and mantissa bits of a float64, as strings."""
    as_int = int(np.array(x, dtype=np.float64).view(np.uint64))
    bits = format(as_int, "064b")
    return bits[0], bits[1:12], bits[12:]

for x in [1.0, 2.0, -2.0, 0.75, 0.1]:
    s, e, f = float_bits(x)
    print(f"{x:6}: sign {s} | exponent {e} (= 2^{int(e, 2) - 1023:3d}) | mantissa {f}")

Look at `0.1`: its mantissa is the repeating pattern `1001 1001 ...`, rounded at the last bit. That rounding is the error you saw above.

52 mantissa bits correspond to about $52 \log_{10} 2 \approx 15.7$ decimal digits. So **a float64 carries about 16 significant digits**, no more.

## 4. Machine epsilon

The gap between 1 and the next larger float is called **machine epsilon**, $\epsilon_m = 2^{-52} \approx 2.2\times10^{-16}$. Any change smaller than about half of that is lost when added to 1. NumPy reports it with `np.finfo`:

In [ ]:
eps = np.finfo(float).eps
print("machine epsilon:", eps, "=", 2.0**-52)
print(1.0 + eps == 1.0)
print(1.0 + eps / 4 == 1.0)    # the small number is lost entirely

You can find $\epsilon_m$ yourself by halving a number until adding it to 1 no longer changes anything:

In [ ]:
e = 1.0
n_halvings = 0
while 1.0 + e / 2 != 1.0:
    e = e / 2
    n_halvings += 1
print(n_halvings, e)

The gap between neighbouring floats is not fixed: it scales with the size of the number. Near $x$, floats are spaced by about $\epsilon_m |x|$, so the **relative** precision is always about $10^{-16}$. `np.spacing(x)` gives the gap at $x$.

In [ ]:
for x in [1.0, 1e3, 1e10, 1e20, 1e-10]:
    print(f"x = {x:8.0e}   spacing = {np.spacing(x):.3e}   relative = {np.spacing(x)/x:.2e}")

At $10^{20}$ the floats are 16384 apart, so adding 1 does nothing at all:

In [ ]:
print(1e20 + 1 - 1e20)

NumPy also has **single precision** (`float32`, 32 bits, about 7 digits). It is faster and uses half the memory, which is why it is common in machine learning and graphics, but it is rarely precise enough for physics calculations. You will meet it again with PyTorch.

In [ ]:
print("float32:", np.finfo(np.float32).eps, "   digits ~", np.finfo(np.float32).precision)
print("float64:", np.finfo(np.float64).eps, "   digits ~", np.finfo(np.float64).precision)
print(np.float32(0.1), f"{np.float32(0.1):.20f}")

### Your turn 1

(a) Use `float_bits` to look at `3.0`, `0.5`, `0.25`, and `1/3`. Which have a "clean" mantissa and which don't? Why?

(b) The Earth–Sun distance is about $1.496\times10^{11}$ m. What is the smallest distance change (in m) that a float64 can represent at that size? Could you represent the position of the Earth to the nearest micrometre in metres measured from the Sun?

In [ ]:
# Your code here

## 5. Overflow, underflow, `inf`, and `nan`

The exponent has a limited range, so floats have a largest and a smallest positive (normal) value:

In [ ]:
info = np.finfo(float)
print("largest float :", info.max)
print("smallest normal positive float:", info.tiny)

- **Overflow:** a result bigger than about $1.8\times10^{308}$ becomes `inf` (infinity).
- **Underflow:** a result smaller than about $10^{-308}$ loses precision gradually, and below about $5\times10^{-324}$ becomes exactly `0.0`.
- Undefined operations like $\infty - \infty$ or $0/0$ give `nan`, "not a number".

NumPy prints a `RuntimeWarning` when this happens but keeps going. A warning is not an error: the cell finishes, but you should always read warnings, because they usually mean a result is garbage.

In [ ]:
print(np.float64(1e308) * 10)
print(np.exp(1000.0))
print(np.float64(1e-300) * 1e-30)
print(np.inf - np.inf)
print(np.array([0.0]) / 0.0)

Plain Python floats behave a little differently: they raise errors instead of warnings for some of these cases.

In [ ]:
import math
try:
    math.exp(1000)
except OverflowError as err:
    print("OverflowError:", err)
try:
    1.0 / 0.0
except ZeroDivisionError as err:
    print("ZeroDivisionError:", err)

**`nan` is contagious and strange.** Any arithmetic with `nan` gives `nan`, and `nan` is not equal to anything, not even itself. Use `np.isnan`, `np.isinf`, or `np.isfinite` to test for them.

In [ ]:
x = np.array([1.0, np.nan, np.inf, -2.0])
print(x + 1)
print(np.nan == np.nan)
print("isnan   :", np.isnan(x))
print("isfinite:", np.isfinite(x))
print("sum     :", np.sum(x), "   nansum:", np.nansum(x[np.isfinite(x)]))

### 5.1 Physics example: Boltzmann factors

**Physics recap.** In thermal equilibrium at temperature $T$, the probability of finding a system in a state of energy $E$ is proportional to the **Boltzmann factor** $e^{-E/k_BT}$. The ratio of populations of two levels separated by $\Delta E$ is $e^{-\Delta E/k_B T}$.

For hydrogen's $n=2$ level ($\Delta E = 10.2$ eV above the ground state) at room temperature, $\Delta E/k_BT \approx 400$. Computing each factor on its own and then taking a ratio is risky. Suppose we want the ratio of the populations of two levels at 10.2 eV and 12.1 eV, each relative to the ground state, in a cool gas at $T = 100$ K:

In [ ]:
kT = const.k * 100.0 / const.e          # k_B T in eV at T = 100 K
E2, E3 = 10.2, 12.1                     # eV
w2 = np.exp(-E2 / kT)
w3 = np.exp(-E3 / kT)
print("individual factors:", w2, w3)
print("naive ratio w3/w2 :", w3 / w2)
print("ratio computed as exp(-(E3-E2)/kT):", np.exp(-(E3 - E2) / kT))
print("log10 of the ratio:", -(E3 - E2) / kT / np.log(10))

Both factors ($e^{-1180}$ and smaller) underflow to zero, so the naive ratio is $0/0$ = `nan`. Combining the exponents first, $e^{-E_3/kT}/e^{-E_2/kT} = e^{-(E_3-E_2)/kT} \approx e^{-220}$, gives a perfectly good (tiny) answer. Keeping just the exponent (or the log of the ratio) works even when the answer itself would underflow. The general trick: **work with logarithms** (or exponents) of huge or tiny quantities, and exponentiate only at the end.

## 6. Comparing floats: never use `==`

Because nearly every float carries a small error, test whether two floats are **close**, not equal. `np.isclose(a, b)` returns `True` if

$$|a - b| \le \texttt{atol} + \texttt{rtol}\,|b|,$$

with default relative tolerance `rtol = 1e-5` and absolute tolerance `atol = 1e-8`. `np.allclose` checks a whole array at once. (`math.isclose` is the single-number version in plain Python, with `rel_tol=1e-9` by default.)

In [ ]:
print(np.isclose(0.1 + 0.2, 0.3))
print(np.isclose(1.0, 1.0 + 1e-7))
print(np.isclose(1.0, 1.001))
print(np.isclose(1.0, 1.001, rtol=1e-2))

**Pitfall:** the absolute tolerance `atol = 1e-8` means that tiny numbers all look "close" to each other and to zero. For physical constants in SI units this can be badly wrong: Planck's constant is $6.6\times10^{-34}$ J s, so `np.isclose` would say it equals zero!

In [ ]:
print(np.isclose(const.h, 0.0))                        # True, which is nonsense
print(np.isclose(const.h, 0.0, atol=0))                # False, as it should be
print(np.isclose(const.h, 6.626e-34, rtol=1e-3, atol=0))

When values are far from 1, set `atol=0` and use a relative tolerance (or work in natural units like eV or nm so numbers are of order 1).

The same rule applies to loops: never write `while t != t_end:` with floating-point `t`, since `t` may step right over `t_end`. Use `while t < t_end:` or loop over an integer counter.

In [ ]:
t = 0.0
steps = 0
while t < 1.0:          # with "t != 1.0" this loop would never end!
    t += 0.1
    steps += 1
print(steps, t)

Notice it took **11** steps, not 10, because after 10 additions `t` is $0.9999999999999999$, which is still less than 1. Looping over an integer counter (`for i in range(10): t = i * 0.1`) avoids this.

## 7. Absolute and relative error

If the true value is $x$ and your computed value is $\tilde x$:

- **absolute error** $= |\tilde x - x|$, in the same units as $x$;
- **relative error** $= |\tilde x - x| / |x|$, dimensionless.

Relative error is usually the meaningful one: an error of 1 mm is superb for the Earth–Moon distance and terrible for the size of an atom. A relative error of $10^{-k}$ means roughly $k$ correct significant digits. Machine epsilon says the best possible relative error for a float64 is about $10^{-16}$.

## 8. Round-off accumulation

Each floating-point operation can introduce a relative error of up to about $\epsilon_m/2$. One error is harmless, but a calculation with millions of steps (like a long simulation) can accumulate many. Let's add 0.1 to itself $N$ times and compare with the exact answer $N/10$.

In [ ]:
def add_repeatedly(value, N):
    total = 0.0
    for i in range(N):
        total += value
    return total

for N in [10, 1000, 100_000, 1_000_000]:
    s = add_repeatedly(0.1, N)
    exact = N / 10
    print(f"N = {N:8d}   sum = {s:.12f}   relative error = {abs(s - exact) / exact:.1e}")

The relative error grows with $N$. Here it grows roughly in proportion to $N$, because the error made in each addition always has the same sign; when the errors are random they tend to partly cancel and the growth is more like $\sqrt{N}$. Either way, a long calculation can lose many of its 16 digits.

Smarter algorithms do better. `np.sum` adds in a tree-like ("pairwise") order, which keeps the error much smaller, and `math.fsum` tracks the lost digits and is essentially exact:

In [ ]:
N = 1_000_000
values = np.full(N, 0.1)
exact = N / 10
for name, s in [("loop", add_repeatedly(0.1, N)),
                ("np.sum", np.sum(values)),
                ("math.fsum", math.fsum(values))]:
    print(f"{name:10s} {s:.10f}   relative error {abs(s - exact) / exact:.1e}")

(`math.fsum` returns the correctly rounded sum of the stored values. Since the stored 0.1 is slightly more than 1/10, it is not guaranteed to hit $N/10$ exactly, but here it does.)

Let's see the growth on a **log-log plot**, the standard way to show how an error depends on a parameter. A power law $\text{error} \propto N^p$ is a straight line of slope $p$.

In [ ]:
Ns = np.unique(np.logspace(1, 6, 25).astype(int))   # integers from 10 to 1e6, evenly spaced in log
rel_err = []
for n in Ns:
    s = add_repeatedly(0.1, n)
    rel_err.append(abs(s - n / 10) / (n / 10))
rel_err = np.array(rel_err)

fig, ax = plt.subplots()
ax.loglog(Ns, rel_err, "o", label="loop sum of 0.1")
ax.loglog(Ns, 1e-17 * Ns, "--", label=r"$\propto N$")
ax.axhline(eps, color="gray", ls=":", label=r"machine $\epsilon$")
ax.set_xlabel("number of additions $N$")
ax.set_ylabel("relative error")
ax.legend()
plt.show()

(`.astype(int)` converts an array to integers, and `np.unique` removes duplicates. Points where the error is exactly zero do not appear on a log plot, since $\log 0 = -\infty$.)

### Your turn 2

Compute the **harmonic sum** $S = \sum_{k=1}^{N} 1/k$ for $N = 10^6$ in two orders: from $k=1$ up to $N$, and from $k=N$ down to 1. Use a loop for each (or `np.sum` on an array; then use `np.cumsum` to compare). Compare both with `math.fsum`. Which order is more accurate, and can you explain why? (Hint: think about adding a tiny number to a large running total.)

In [ ]:
# Your code here

## 9. Catastrophic cancellation

The most dangerous situation is **subtracting two nearly equal numbers**. Each number may carry 16 good digits, but if they agree in their first 12 digits, the difference has only about 4 good digits left. The relative error of the result is roughly

$$\frac{\epsilon_m \cdot |a|}{|a - b|},$$

which explodes as $a \to b$. This is called **catastrophic cancellation**. The cure is almost never "more digits"; it is to **rewrite the formula** so the subtraction does not happen.

### 9.1 $1 - \cos x$ for small $x$

For small $x$, $\cos x \approx 1 - x^2/2$, so $1 - \cos x \approx x^2/2$. But computing `1 - np.cos(x)` subtracts two numbers that are both very close to 1.

In [ ]:
for x in [1e-1, 1e-4, 1e-7, 1e-8, 1e-9]:
    naive = 1 - np.cos(x)
    print(f"x = {x:.0e}   1 - cos(x) = {naive:.6e}   x^2/2 = {x**2 / 2:.6e}")

At $x = 10^{-8}$ the answer should be $5\times10^{-17}$, but $\cos x$ rounds to exactly 1, so the naive formula gives 0: a 100% error.

The fix uses the identity $1 - \cos x = 2\sin^2(x/2)$, which involves no subtraction at all:

In [ ]:
x = np.logspace(-10, 0, 200)
naive = 1 - np.cos(x)
stable = 2 * np.sin(x / 2)**2

rel_err = np.abs(naive - stable) / stable
fig, ax = plt.subplots()
ax.loglog(x, rel_err, label=r"naive $1-\cos x$")
ax.loglog(x, eps / x**2, "--", label=r"estimate $\epsilon_m / x^2$")
ax.set_xlabel("$x$")
ax.set_ylabel("relative error")
ax.set_ylim(1e-17, 10)
ax.legend()
plt.show()

The error estimate follows from the formula above: $|a| \approx 1$ and $|a - b| \approx x^2/2$, so the relative error is about $\epsilon_m/(x^2/2) \sim \epsilon_m/x^2$. On a log-log plot that is a straight line of slope $-2$, and the naive result hits 100% error around $x \sim 10^{-8} = \sqrt{\epsilon_m}$. (Below that, the naive value is exactly 0, a relative error of exactly 1.)

### 9.2 Kinetic energy of a slow particle: $\gamma - 1$

**Physics recap.** The relativistic kinetic energy is $K = (\gamma - 1) mc^2$ with $\gamma = 1/\sqrt{1-\beta^2}$ and $\beta = v/c$. For slow particles, $\gamma$ is barely above 1, and expanding in $\beta$:

$$\gamma - 1 = \tfrac12\beta^2 + \tfrac38\beta^4 + \ldots \quad\Rightarrow\quad K \approx \tfrac12 mv^2 .$$

So computing $\gamma$ and then subtracting 1 is exactly the cancellation problem. Consider an electron moving at 1 m/s ($\beta \approx 3\times10^{-9}$), the drift speed scale of electrons in a wire:

In [ ]:
m_e, c = const.m_e, const.c
v = 1.0                               # m/s
beta = v / c
gamma = 1 / np.sqrt(1 - beta**2)
K_naive = (gamma - 1) * m_e * c**2
K_classical = 0.5 * m_e * v**2
print("gamma       =", gamma)
print("K naive     =", K_naive, "J")
print("K classical =", K_classical, "J")

$\gamma$ rounds to exactly 1, so the relativistic formula claims a moving electron has **zero** kinetic energy. The classical answer is correct to about 1 part in $10^{17}$ here, but we would like one formula that works at all speeds.

**Rewrite to remove the subtraction.** Let $s = \sqrt{1-\beta^2}$. Then

$$\gamma - 1 = \frac{1}{s} - 1 = \frac{1 - s}{s} = \frac{(1-s)(1+s)}{s(1+s)} = \frac{1 - s^2}{s(1+s)} = \frac{\beta^2}{\sqrt{1-\beta^2}\,\left(1+\sqrt{1-\beta^2}\right)}.$$

The last form has no subtraction of nearly equal numbers ($1-\beta^2$ is close to 1, but that is fine: it is $\beta^2$ itself that carries the information, and we keep it intact in the numerator). It is exact algebra, so it is correct at all speeds.

In [ ]:
def gamma_minus_1_naive(beta):
    return 1 / np.sqrt(1 - beta**2) - 1

def gamma_minus_1_stable(beta):
    s = np.sqrt(1 - beta**2)
    return beta**2 / (s * (1 + s))

for b in [0.9, 1e-2, 1e-5, 1e-8]:
    print(f"beta = {b:.0e}   naive = {gamma_minus_1_naive(b):.10e}   stable = {gamma_minus_1_stable(b):.10e}")

Now plot the electron's kinetic energy computed both ways over a huge range of speeds, on log-log axes.

In [ ]:
beta = np.logspace(-10, -0.01, 400)          # from 1e-10 up to about 0.98
mc2_eV = m_e * c**2 / const.e                # electron rest energy in eV
K_naive = gamma_minus_1_naive(beta) * mc2_eV
K_stable = gamma_minus_1_stable(beta) * mc2_eV

fig, ax = plt.subplots()
ax.loglog(beta, K_stable, label="stable formula", lw=2)
ax.loglog(beta, K_naive, ".", ms=3, label=r"naive $(\gamma-1)mc^2$")
ax.loglog(beta, 0.5 * mc2_eV * beta**2, "k:", label=r"classical $\frac{1}{2}mv^2$")
ax.set_xlabel(r"$\beta = v/c$")
ax.set_ylabel("electron kinetic energy (eV)")
ax.legend()
plt.show()

Below $\beta \approx 10^{-8}$ the naive points vanish (the value is exactly zero, which a log axis cannot show), and well before that they become a noisy staircase. The relative error makes the damage clear:

In [ ]:
rel_err = np.abs(K_naive - K_stable) / K_stable

fig, ax = plt.subplots()
ax.loglog(beta, rel_err, label="naive vs stable")
ax.loglog(beta, eps / beta**2, "--", label=r"$\epsilon_m / \beta^2$")
ax.set_ylim(1e-17, 10)
ax.set_xlabel(r"$\beta$")
ax.set_ylabel("relative error of naive formula")
ax.legend()
plt.show()

Same pattern as $1-\cos x$: the relative error is about $\epsilon_m/\beta^2$, reaching 100% at $\beta \sim \sqrt{\epsilon_m} \approx 10^{-8}$. Even at $\beta = 10^{-4}$ (about 30 km/s, faster than any spacecraft), the naive formula has only about 8 correct digits.

### 9.3 Library functions for small arguments: `expm1` and `log1p`

Two combinations appear so often that NumPy provides accurate built-in versions:

- `np.expm1(x)` $= e^x - 1$, accurate even when $x$ is tiny (you used this in the Planck function);
- `np.log1p(x)` $= \ln(1 + x)$, accurate even when $x$ is tiny.

They can be combined to write $\gamma - 1$ another stable way. Since $\gamma = (1-\beta^2)^{-1/2} = \exp\!\left[-\tfrac12 \ln(1-\beta^2)\right]$,

$$\gamma - 1 = \texttt{expm1}\!\left(-\tfrac12\,\texttt{log1p}(-\beta^2)\right).$$

In [ ]:
x = 1e-12
print("exp(x) - 1  :", np.exp(x) - 1, "   expm1(x):", np.expm1(x))
print("log(1 + x)  :", np.log(1 + x), "   log1p(x):", np.log1p(x))

b = 1e-9
print("gamma-1 via expm1/log1p:", np.expm1(-0.5 * np.log1p(-b**2)))
print("gamma-1 stable formula :", gamma_minus_1_stable(b))

### 9.4 The quadratic formula

The roots of $ax^2 + bx + c = 0$ are

$$x_\pm = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}.$$

If $b^2 \gg |4ac|$, then $\sqrt{b^2-4ac} \approx |b|$, and one of the two roots subtracts nearly equal numbers. Example: $x^2 + 10^8 x + 1 = 0$, whose roots are very close to $-10^8$ and $-10^{-8}$.

In [ ]:
a, b, c = 1.0, 1e8, 1.0
disc = np.sqrt(b**2 - 4 * a * c)
x_plus = (-b + disc) / (2 * a)
x_minus = (-b - disc) / (2 * a)
print("x+ =", x_plus, "  x- =", x_minus)
print("check: a x+^2 + b x+ + c =", a * x_plus**2 + b * x_plus + c)

The small root should be about $-1.0\times10^{-8}$; the naive result is off by about 25%, and plugging it back in gives a residual near 0.25 instead of 0.

**Stable version.** Compute the root that involves an *addition* of same-sign numbers first, then get the other from the product of roots, $x_+ x_- = c/a$ (Vieta's formula):

$$q = -\tfrac12\left(b + \operatorname{sign}(b)\sqrt{b^2-4ac}\right), \qquad x_1 = \frac{q}{a}, \qquad x_2 = \frac{c}{q}.$$

In [ ]:
def quadratic_roots_stable(a, b, c):
    """Real roots of a x^2 + b x + c = 0, avoiding cancellation. Assumes b^2 >= 4ac and b != 0."""
    q = -0.5 * (b + np.sign(b) * np.sqrt(b**2 - 4 * a * c))
    return q / a, c / q

x1, x2 = quadratic_roots_stable(1.0, 1e8, 1.0)
print("x1 =", x1, "  x2 =", x2)
print("check x2:", 1.0 * x2**2 + 1e8 * x2 + 1.0)

### Your turn 3

The function $f(x) = \sqrt{x^2 + 1} - x$ suffers from cancellation for large $x$.

(a) Evaluate it naively for $x = 10^2, 10^4, 10^6, 10^8, 10^{10}$ and see what happens.

(b) Multiply and divide by $\sqrt{x^2+1} + x$ to find an equivalent formula without the subtraction, and evaluate that.

(c) Plot the relative error of the naive version (relative to your stable version) vs $x$ on log-log axes for $1 \le x \le 10^{10}$, along with a guess for how the error scales.

In [ ]:
# Your code here

### Your turn 4

A photon of energy $E$ Compton-scatters off an electron at angle $\theta$; the change in wavelength is $\Delta\lambda = \lambda_C (1 - \cos\theta)$ with $\lambda_C = h/(m_e c) \approx 2.43$ pm. For a grazing angle of $\theta = 10^{-9}$ rad, compute $\Delta\lambda$ naively and with a stable formula. How large is the relative error of the naive result? (Use `const.h`, `const.m_e`, `const.c`.)

In [ ]:
# Your code here

## 10. Truncation error vs round-off error

So far all errors came from storing numbers with finite precision: **round-off error**. Numerical methods have a second, completely different kind of error. Most methods replace an exact but infinite mathematical process (an infinite series, a limit, an integral) by a finite approximation. The error from that approximation is called **truncation error** (or *approximation error*). It exists even with infinitely precise arithmetic.

The simplest example is a truncated Taylor series. Recall

$$e^x = \sum_{k=0}^{\infty} \frac{x^k}{k!} = 1 + x + \frac{x^2}{2} + \frac{x^3}{6} + \ldots$$

Keeping only the terms up to $k = n$ leaves out the rest. For small $x$, the first omitted term $x^{n+1}/(n+1)!$ estimates the truncation error.

In [ ]:
def exp_taylor(x, n_terms):
    """Approximate e^x by summing the Taylor series terms k = 0 .. n_terms-1."""
    total = 0.0
    term = 1.0                      # the k = 0 term
    for k in range(n_terms):
        total += term
        term = term * x / (k + 1)   # x^(k+1)/(k+1)! from x^k/k!
    return total

x = 1.0
for n in [1, 2, 4, 8, 12, 16, 20]:
    approx = exp_taylor(x, n)
    print(f"{n:2d} terms: {approx:.16f}   relative error {abs(approx - np.e) / np.e:.1e}")

(Notice the trick of getting each term from the previous one, instead of computing a power and a factorial separately; this is cheaper and avoids huge intermediate numbers.)

Plot the error versus the number of terms:

In [ ]:
n_terms = np.arange(1, 26)
fig, ax = plt.subplots()
for x in [0.5, 1.0, 5.0]:
    err = [abs(exp_taylor(x, n) - np.exp(x)) / np.exp(x) for n in n_terms]
    ax.semilogy(n_terms, err, "o-", ms=4, label=f"$x = {x}$")
ax.axhline(eps, color="gray", ls=":", label=r"machine $\epsilon$")
ax.set_xlabel("number of terms kept")
ax.set_ylabel(r"relative error in $e^x$")
ax.legend()
plt.show()

(The line `err = [... for n in n_terms]` is a *list comprehension*, a compact way to build a list with a loop, from notebook 02.)

Two regimes are visible:

1. **Truncation-dominated:** the error drops rapidly as you keep more terms.
2. **Round-off floor:** once the error reaches about $\epsilon_m \sim 10^{-16}$, adding more terms does not help; the answer is as good as float64 allows.

### 10.1 When round-off wins: $e^{-x}$ for large $x$

For $x = -20$, the Taylor series has terms as large as $20^{20}/20! \approx 4\times10^{7}$ with alternating signs, which must cancel to give $e^{-20} \approx 2\times10^{-9}$. Each large term carries an absolute round-off error of about $4\times10^7\times 10^{-16} = 4\times10^{-9}$, as large as the answer itself.

In [ ]:
x = -20.0
for n in [20, 40, 60, 80, 100]:
    approx = exp_taylor(x, n)
    print(f"{n:3d} terms: {approx: .6e}   exact {np.exp(x):.6e}")
print("better: 1 / exp_taylor(20, 100) =", 1 / exp_taylor(20.0, 100))

No number of terms fixes it, because the problem is cancellation, not truncation. The fix is again to rewrite: $e^{-20} = 1/e^{20}$, where all terms are positive.

### 10.2 Looking ahead

In the next notebook you will approximate a derivative by $f'(x) \approx [f(x+h) - f(x)]/h$. Its truncation error shrinks as the step $h$ gets smaller, but the numerator subtracts two nearly equal numbers, so its round-off error *grows* as $h$ gets smaller. The total error therefore has a minimum at some optimal $h$, and you will find it with exactly the kind of log-log error plot you made here.

### Your turn 5

The Taylor series for $\sin x$ is $x - x^3/3! + x^5/5! - \ldots$. Write a function `sin_taylor(x, n_terms)` (build each term from the previous one: the next term is the current one times $-x^2/[(2k+2)(2k+3)]$ for $k = 0, 1, 2, \dots$). Plot the relative error vs number of terms for $x = 0.5$, $x = 3$, and $x = 30$ on a semilog plot. Explain the behavior at $x = 30$, and suggest how to get an accurate $\sin(30)$ using the periodicity of $\sin$.

In [ ]:
# Your code here

## Summary

- Integers are exact (NumPy `int64` can silently wrap around past $\approx 9.2\times10^{18}$). Most decimals, like 0.1, are **not** exactly representable in binary.
- A float64 = sign (1 bit) + exponent (11 bits) + mantissa (52 bits): about **16 significant digits**, range about $10^{-308}$ to $10^{308}$.
- **Machine epsilon** `np.finfo(float).eps` $\approx 2.2\times10^{-16}$ is the relative spacing of floats; `np.spacing(x)` is the gap near $x$.
- Overflow gives `inf`, underflow gives `0.0`, undefined results give `nan`. Test with `np.isnan`, `np.isinf`, `np.isfinite`; use logarithms for huge/tiny quantities.
- Never compare floats with `==`. Use `np.isclose`/`np.allclose` (and `atol=0` for numbers far from 1).
- Absolute error $|\tilde x - x|$; relative error $|\tilde x - x|/|x|$. Show error vs a parameter on a **log-log plot**: power laws are straight lines.
- **Round-off error** accumulates over many operations; `np.sum` and `math.fsum` add more carefully than a loop.
- **Catastrophic cancellation:** subtracting nearly equal numbers destroys digits. Rewrite the formula: $1-\cos x = 2\sin^2(x/2)$; $\gamma-1 = \beta^2/[\sqrt{1-\beta^2}(1+\sqrt{1-\beta^2})]$; stable quadratic roots via $q$ and $c/q$; `np.expm1`, `np.log1p`.
- **Truncation error** comes from approximating an infinite process with a finite one (e.g. a truncated Taylor series). The total error of a numerical method is truncation + round-off, and the two often trade off against each other.

## Further reading

- M. Newman, *Computational Physics*, Section 4.2 "Numerical error" (Chapter 4 is freely available on the author's website).
- Python tutorial: [Floating-point arithmetic: issues and limitations](https://docs.python.org/3/tutorial/floatingpoint.html).
- D. Goldberg, "What every computer scientist should know about floating-point arithmetic" (1991), freely available online; advanced but a classic.